# Toxic Comments Dataset

Our necessary libraries:
sci-kit learn,
pandas,
nltk

In [ ]:
!pip install pandas scikit-learn nltk

### We load the dataset from my Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/toxic_comments_train.csv")

df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [ ]:
df.shape

(159571, 8)

In [ ]:
labels = ["toxic","severe_toxic","obscene","threat","insult","identity_hate"]

df[labels].sum()
df[labels].mean()

,0
toxic,0.095844
severe_toxic,0.009996
obscene,0.052948
threat,0.002996
insult,0.049364
identity_hate,0.008805


### Note: As we can see, the dataset is highly imbalanced. Most comments are non toxic, while categories such as "threat" or "identity_hate" appear in less than 1% of the data.

# Preprocessing

In [ ]:
import re
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

#lowercase
df["text"] = df["comment_text"].fillna("").str.lower()

#we remove URLs
df["text"] = df["text"].apply(lambda x: re.sub(r"http\S+|www\S+", "", x))

#numbers
df["text"] = df["text"].apply(lambda x: re.sub(r"\d+", "", x))

#punctuation
df["text"] = df["text"].apply(
    lambda x: x.translate(str.maketrans('', '', string.punctuation))
)

#tokenization
df["tokens"] = df["text"].apply(lambda x: x.split())

#stopwords
df["tokens"] = df["tokens"].apply(
    lambda tokens: [word for word in tokens if word not in ENGLISH_STOP_WORDS]
)

#we remove short words
df["tokens"] = df["tokens"].apply(
    lambda tokens: [word for word in tokens if len(word) > 2]
)

df[["comment_text", "text", "tokens"]].head()

,comment_text,text,tokens
0,Explanation\nWhy the edits made under my usern...,explanation\nwhy the edits made under my usern...,"[explanation, edits, username, hardcore, metal..."
1,D'aww! He matches this background colour I'm s...,daww he matches this background colour im seem...,"[daww, matches, background, colour, seemingly,..."
2,"Hey man, I'm really not trying to edit war. It...",hey man im really not trying to edit war its j...,"[hey, man, really, trying, edit, war, just, gu..."
3,"""\nMore\nI can't make any real suggestions on ...",\nmore\ni cant make any real suggestions on im...,"[make, real, suggestions, improvement, wondere..."
4,"You, sir, are my hero. Any chance you remember...",you sir are my hero any chance you remember wh...,"[sir, hero, chance, remember, page, thats]"


## POS Tagging

In [ ]:
import nltk

nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [ ]:
from nltk import pos_tag

df["pos_tags"] = df["tokens"].apply(lambda tokens: pos_tag(tokens))
df["pos_tags"].head()

,pos_tags
0,"[(explanation, NN), (edits, NNS), (username, J..."
1,"[(daww, NN), (matches, NNS), (background, IN),..."
2,"[(hey, NN), (man, NN), (really, RB), (trying, ..."
3,"[(make, VB), (real, JJ), (suggestions, NNS), (..."
4,"[(sir, NN), (hero, NN), (chance, NN), (remembe..."


## Lemmatization

In [ ]:
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith("J"):
        return wordnet.ADJ
    elif treebank_tag.startswith("V"):
        return wordnet.VERB
    elif treebank_tag.startswith("N"):
        return wordnet.NOUN
    elif treebank_tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def lemmatize_with_pos(tagged_tokens):
    return [lemmatizer.lemmatize(word, get_wordnet_pos(tag))
            for word, tag in tagged_tokens]

df["lemmas"] = df["pos_tags"].apply(lemmatize_with_pos)

df["clean_text"] = df["lemmas"].apply(lambda tokens: " ".join(tokens))

df[["comment_text", "clean_text"]].head()

,comment_text,clean_text
0,Explanation\nWhy the edits made under my usern...,explanation edits username hardcore metallica ...
1,D'aww! He matches this background colour I'm s...,daww match background colour seemingly stuck t...
2,"Hey man, I'm really not trying to edit war. It...",hey man really try edit war just guy constantl...
3,"""\nMore\nI can't make any real suggestions on ...",make real suggestion improvement wonder sectio...
4,"You, sir, are my hero. Any chance you remember...",sir hero chance remember page thats


# Train-test split

### We split the data into train and test sets to evaluate the model on unseen examples and avoid overfitting

In [ ]:
from sklearn.model_selection import train_test_split

labels = ["toxic","severe_toxic","obscene","threat","insult","identity_hate"]

X = df["clean_text"]
y = df[labels]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(127656,) (31915,)
(127656, 6) (31915, 6)


## Text representation, TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=2
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape, X_test_tfidf.shape)

(127656, 20000) (31915, 20000)


## Multi label model, One Vs Rest and Logistic regression

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

model = OneVsRestClassifier(
    LogisticRegression(max_iter=1000)
)

model.fit(X_train_tfidf, y_train)

OneVsRestClassifier(estimator=LogisticRegression(max_iter=1000))

### Note: This is a multi label classification problem, meaning that each comment can belong to more than one category at the same time (example: both "toxic" and "insult"). To handle this, we use the One VS Rest strategy. This strategy trains one binary classifier for each label, learning to distinguish between the presence and absence of that specific label.

### As base classifier, we use Logistic regression because the text has already been transformed into numerical vectores using TF-IDF. Also because Logistic regression is well suited for high dimensional sparse data such as TF-IDF features.


## Predictions

#### We predict labels for the test set using the trained model

In [ ]:
y_pred = model.predict(X_test_tfidf)
y_pred.shape

(31915, 6)

### Note: the prediction matrix has shape of: number of test samples and numbers of labels. So, each row corresponds to one comment and each column corresponds to one label. The model predicts 0 or 1 for each label.

# Evaluation

In [ ]:
from sklearn.metrics import f1_score, hamming_loss, classification_report

print("F1 micro:", f1_score(y_test, y_pred, average="micro"))
print("F1 macro:", f1_score(y_test, y_pred, average="macro"))
print("Hamming loss:", hamming_loss(y_test, y_pred))

print("\nPer-label report:\n")
print(classification_report(y_test, y_pred, target_names=labels, zero_division=0))

F1 micro: 0.6733825445071646
F1 macro: 0.4985943497939303
Hamming loss: 0.01964071230873675

Per-label report:

               precision    recall  f1-score   support

        toxic       0.91      0.61      0.73      3056
 severe_toxic       0.54      0.25      0.34       321
      obscene       0.91      0.62      0.74      1715
       threat       0.65      0.18      0.28        74
       insult       0.82      0.49      0.61      1614
identity_hate       0.75      0.18      0.29       294

    micro avg       0.87      0.55      0.67      7074
    macro avg       0.76      0.39      0.50      7074
 weighted avg       0.86      0.55      0.67      7074
  samples avg       0.06      0.05      0.05      7074



### Note: the model achieves an f1 micro score of 0.67 and f1 macro score of 0.50. The higher micro score shows that the model performs better on more frequent labels. The lower macro score indicates that rare classes are harder to predict. The hamming loss is low, around 0.02, which means that most labels are predicted correctly. Considering the strong class imbalance, the model shows reasonable performance.

# A few examples

In [ ]:
import numpy as np

idx = np.random.choice(len(X_test), size=20, replace=False)
sample_texts = X_test.iloc[idx].values
sample_true = y_test.iloc[idx].values
sample_pred = y_pred[idx]

for i in range(20):
    print("TEXT:", sample_texts[i][:200], "...")
    print("TRUE:", dict(zip(labels, sample_true[i])))
    print("PRED:", dict(zip(labels, sample_pred[i])))
    print("-"*80)

TEXT: clear vandalism removal relevant information ...
TRUE: {'toxic': np.int64(0), 'severe_toxic': np.int64(0), 'obscene': np.int64(0), 'threat': np.int64(0), 'insult': np.int64(0), 'identity_hate': np.int64(0)}
PRED: {'toxic': np.int64(0), 'severe_toxic': np.int64(0), 'obscene': np.int64(0), 'threat': np.int64(0), 'insult': np.int64(0), 'identity_hate': np.int64(0)}
--------------------------------------------------------------------------------
TEXT: suggest permabanning use troll wikipedia need remove ...
TRUE: {'toxic': np.int64(0), 'severe_toxic': np.int64(0), 'obscene': np.int64(0), 'threat': np.int64(0), 'insult': np.int64(0), 'identity_hate': np.int64(0)}
PRED: {'toxic': np.int64(0), 'severe_toxic': np.int64(0), 'obscene': np.int64(0), 'threat': np.int64(0), 'insult': np.int64(0), 'identity_hate': np.int64(0)}
--------------------------------------------------------------------------------
TEXT: redirect talksri balaji temple nagar ...
TRUE: {'toxic': np.int64(0), 'severe_toxi

### Note: we randomly inspect a few test samples to qualitatively verify the model's predictions. This helps confirm that the classifier behaves consistently with the ground truth labels. From these examples, we can observe that the model correctly identifies clearly toxic comments and also correctly predicts many non toxic cases.

## Conclusion: overall, the baseline performs reasonably well given the strong class imbalance. Performance is higher for frequent labels like "toxic" or "obscene" and lower for rare labels such as "threat".